# บทที่ 01: จากข้อมูลราคา สู่สัญญาณแรก

Robo Trade · Python + Webull OpenAPI

Notebook นี้รันแบบออฟไลน์ ข้อมูล DEMO สร้างขึ้นเอง 120 แท่ง ไม่ใช่ราคาหุ้นจริง ไม่มีคำสั่งซื้อขาย และไม่รายงานผลตอบแทน

เป้าหมาย: เข้าใจ SMA, สถานะเป้าหมาย, warm-up และเวลาเกิดสัญญาณ

## 1. เตรียมเครื่องมือ

ติดตั้ง pandas==2.3.2 ใน environment ของคุณก่อนรัน ใช้ Kernel ใหม่แล้วรันจากบนลงล่างได้ ข้อมูลตัวอย่างฝังอยู่ในไฟล์นี้

In [1]:
from io import StringIO
import pandas as pd
print('pandas', pd.__version__)

pandas 2.3.2


## 2. ข้อมูลราคา

แต่ละแถวเป็นหนึ่งแท่งรายวันสมมติ ไม่มีปฏิทินวันซื้อขายจริง หน่วยราคาคือ USD ในเชิงภาพประกอบ เราจะใช้ชุดเดียวกับห้องทดลองบนเว็บ

In [2]:
csv_text = 'day,close\n1,100.00\n2,101.44\n3,102.72\n4,103.71\n5,104.35\n6,104.64\n7,104.67\n8,104.55\n9,104.45\n10,104.51\n11,104.80\n12,105.36\n13,106.15\n14,107.04\n15,107.90\n16,108.56\n17,108.89\n18,108.84\n19,108.38\n20,107.62\n21,106.66\n22,105.69\n23,104.84\n24,104.24\n25,103.93\n26,103.89\n27,104.03\n28,104.22\n29,104.30\n30,104.14\n31,103.68\n32,102.90\n33,101.87\n34,100.71\n35,99.60\n36,98.68\n37,98.08\n38,97.87\n39,98.04\n40,98.49\n41,99.10\n42,99.72\n43,100.20\n44,100.44\n45,100.42\n46,100.19\n47,99.84\n48,99.54\n49,99.42\n50,99.61\n51,100.19\n52,101.13\n53,102.37\n54,103.76\n55,105.14\n56,106.36\n57,107.31\n58,107.92\n59,108.23\n60,108.32\n61,108.33\n62,108.42\n63,108.70\n64,109.26\n65,110.10\n66,111.16\n67,112.31\n68,113.39\n69,114.26\n70,114.78\n71,114.89\n72,114.63\n73,114.06\n74,113.33\n75,112.61\n76,112.02\n77,111.68\n78,111.61\n79,111.78\n80,112.07\n81,112.34\n82,112.45\n83,112.27\n84,111.75\n85,110.89\n86,109.77\n87,108.54\n88,107.35\n89,106.36\n90,105.69\n91,105.39\n92,105.43\n93,105.71\n94,106.10\n95,106.45\n96,106.63\n97,106.55\n98,106.21\n99,105.68\n100,105.06\n101,104.53\n102,104.22\n103,104.26\n104,104.70\n105,105.51\n106,106.61\n107,107.86\n108,109.09\n109,110.14\n110,110.94\n111,111.43\n112,111.65\n113,111.73\n114,111.78\n115,111.97\n116,112.40\n117,113.15\n118,114.20\n119,115.46\n120,116.80\n'
data = pd.read_csv(StringIO(csv_text))
assert len(data) == 120
assert data['day'].is_unique and data['day'].is_monotonic_increasing
assert data['close'].notna().all()
print(data.head().to_string(index=False))

 day  close
   1 100.00
   2 101.44
   3 102.72
   4 103.71
   5 104.35


## 3. ตรวจด้วยมือก่อนเขียนกลยุทธ์

SMA คือผลรวมราคาปิด n แท่งล่าสุดหารด้วย n โดยต้องมีข้อมูลครบช่วงก่อน

In [3]:
example = [100, 102, 101, 103, 104]
print('SMA 5 =', sum(example) / len(example))
assert sum(example) / len(example) == 102

SMA 5 = 102.0


## 4. กติกา Long / Cash

SMA สั้นอยู่เหนือ SMA ยาว: target=1; กรณีอื่น: target=0
ช่วงข้อมูลไม่ครบเส้นยาวเป็น warm-up เป้าหมายยังเป็นศูนย์ จุดเปลี่ยนแรกหลัง warm-up นับเมื่อเป้าหมายเปลี่ยนจากศูนย์เป็นหนึ่ง

In [4]:
def build_signals(data, short=5, long=20):
    import math
    if not isinstance(short, int) or not isinstance(long, int) or not 1 <= short < long:
        raise ValueError("Require 1 <= short < long")
    out = data.copy()
    out["close"] = pd.to_numeric(out["close"], errors="raise")
    if not out["close"].map(math.isfinite).all() or len(out) < long:
        raise ValueError("Missing prices or insufficient history")
    out["sma_short"] = out["close"].rolling(short).mean()
    out["sma_long"] = out["close"].rolling(long).mean()
    ready = out["sma_long"].notna()
    out["target"] = (
        (out["sma_short"] > out["sma_long"]) & ready
    ).astype(int)
    # A close-based decision can only be considered afterwards.
    out["next_open_target"] = out["target"].shift(1).fillna(0).astype(int)
    out["changed"] = out["target"].diff().fillna(0).ne(0) & ready
    return out

In [5]:
signals = build_signals(data, short=5, long=20)
print(signals.tail(5).to_string(index=False))
print('จุดเปลี่ยนสถานะ:', signals.loc[signals['changed'], 'day'].tolist())
assert signals.loc[signals['changed'], 'day'].tolist() == [20, 24, 52, 80, 108]

 day  close  sma_short  sma_long  target  next_open_target  changed
 116 112.40    111.906  108.1160       1                 1    False
 117 113.15    112.206  108.4460       1                 1    False
 118 114.20    112.700  108.8455       1                 1    False
 119 115.46    113.436  109.3345       1                 1    False
 120 116.80    114.402  109.9215       1                 1    False
จุดเปลี่ยนสถานะ: [20, 24, 52, 80, 108]


## 5. เวลาเป็นส่วนหนึ่งของกติกา

หลังรู้ราคาปิดแท่ง t แล้วจึงรู้ target[t] เป้าหมายนี้ใช้พิจารณาได้ภายหลัง ไม่ใช่ย้อนไปซื้อที่ราคาเปิดแท่ง t

next_open_target เป็นการเลื่อนเป้าหมายหนึ่งแท่ง ไม่ใช่หลักฐานว่ามี order หรือ fill

In [6]:
print(signals.iloc[17:23][['day','close','target','next_open_target']].to_string(index=False))
assert signals.loc[19, 'target'] == 1
assert signals.loc[19, 'next_open_target'] == 0
assert signals.loc[20, 'next_open_target'] == 1

 day  close  target  next_open_target
  18 108.84       0                 0
  19 108.38       0                 0
  20 107.62       1                 0
  21 106.66       1                 1
  22 105.69       1                 1
  23 104.84       1                 1


## 6. เปลี่ยนหนึ่งอย่าง แล้ววัดผล

เปรียบเทียบจุดเปลี่ยนโดยใช้ข้อมูลชุดเดิม อย่าสรุปว่าจำนวนสัญญาณน้อยกว่าแปลว่ากำไรมากกว่า

In [7]:
for short, long in [(5,20),(5,40),(15,50)]:
    trial = build_signals(data, short, long)
    print(short, long, 'changes =', int(trial['changed'].sum()), 'bars =', trial.loc[trial['changed'],'day'].tolist())

5 20 changes = 5 bars = [20, 24, 52, 80, 108]
5 40 changes = 3 bars = [55, 89, 110]
15 50 changes = 3 bars = [61, 96, 118]


## 7. ตรวจว่าไม่ได้ใช้ข้อมูลอนาคต

ผลคำนวณในอดีตต้องไม่เปลี่ยนเมื่อเราเติมแถวอนาคตเข้ามา

In [8]:
past = build_signals(data.iloc[:70].copy(), 5, 20)
pd.testing.assert_frame_equal(past, signals.iloc[:70])
print('ผ่าน: การเพิ่มข้อมูลอนาคตไม่เปลี่ยนสัญญาณในอดีต')

ผ่าน: การเพิ่มข้อมูลอนาคตไม่เปลี่ยนสัญญาณในอดีต


## 8. รูปแบบข้อมูลจาก Webull

ตัวอย่างถัดไปเป็น JSON จำลองที่มีโครงสร้างตาม Historical Bars API ไม่ใช่ผลเรียก API จริง

แปลง time เป็น UTC และ OHLCV จาก string เป็นตัวเลข ตรวจข้อมูลซ้ำและค่าว่าง แล้วเรียงเวลา

In [9]:
payload = {'result':[{'symbol':'DEMO','instrument_id':'SYNTHETIC','result':[{'time':'2026-01-05T21:00:00+0000','open':'100.00','high':'101.20','low':'99.60','close':'100.80','volume':'120000'}]}]}
stock = next(x for x in payload['result'] if x['symbol']=='DEMO')
bars = pd.DataFrame(stock['result'])
bars['time'] = pd.to_datetime(bars['time'], utc=True, errors='raise')
for name in ['open','high','low','close','volume']:
    bars[name] = pd.to_numeric(bars[name], errors='raise')
assert not bars['time'].duplicated().any()
assert not bars.isna().any().any()
bars = bars.sort_values('time').reset_index(drop=True)
print(bars.to_string(index=False))

                     time  open  high  low  close  volume
2026-01-05 21:00:00+00:00 100.0 101.2 99.6  100.8  120000


## 9. การเชื่อมต่อ Webull เป็นขั้นตอนเพิ่มเติม

โค้ดนี้ตรวจชื่อเมธอดกับ SDK 3.0.0 แต่ยังไม่ได้เชื่อมต่อบัญชีของคุณ ใช้กุญแจเฉพาะบนเครื่อง Python และทำ 2FA กับ Webull เมื่อถูกขอ

ในชุดไฟล์ครบเล่ม ตัวเชื่อมเสริมอยู่ที่ python/webull_bars.py ติดตั้งส่วนเสริมด้วย python -m pip install -r requirements-webull.txt ก่อน แล้ว ตั้ง WEBULL_APP_KEY และ WEBULL_APP_SECRET โดยไม่เผยในประวัติคำสั่ง แล้วรัน:

```text
python python/webull_bars.py --fetch --environment uat --output webull_bars.json
```

UAT: th-api.uat.webullbroker.com
Production: api.webull.co.th

ใช้ Market Data entitlement สำหรับ OpenAPI แยกจากสิทธิ์ในแอป และตรวจเวลาปิดแท่งจริง การตั้ง real_time_required=True เพียงอย่างเดียวไม่ทดแทนการตรวจ session/calendar

## แบบฝึกหัด

1. ใช้ SMA 5/40 แล้วเปรียบเทียบกับ 5/20: warm-up เปลี่ยนไปกี่แท่ง? ยกวันเปลี่ยนสถานะสองวัน
2. ทำไมไม่ควรเรียก target ว่า position ที่ถือจริง?
3. ลองป้อนราคาคงที่ทุกแท่ง เส้นเฉลี่ยควรเท่ากัน และกติกาควรให้สถานะอะไร?

เฉลยย่อ: warm-up ก่อนมีค่าครบเพิ่มจาก 19 เป็น 39 แท่ง; target ยังไม่ผ่านขั้นตอน order/fill; ราคาคงที่ให้ CASH ตามกติกาที่กำหนด

ยังไม่มีการจำลองต้นทุน การจับคู่ หรือผลตอบแทน จึงยังตัดสินความสามารถทำกำไรไม่ได้

## แหล่งอ้างอิง

- Yves Hilpisch, Python for Algorithmic Trading, บท 3-4; https://github.com/yhilpisch/py4at
- https://developer.webull.co.th/apis/docs/sdk
- https://developer.webull.co.th/apis/docs/authentication/overview
- https://developer.webull.co.th/apis/docs/reference/trade-api/historical-bars
- https://github.com/webull-inc/webull-openapi-python-sdk

บทเรียนและโค้ดเขียนขึ้นใหม่ ไม่มีโค้ดหรือ PDF ต้นฉบับหนังสือแนบมา ตรวจเอกสาร 10 กันยายน 2026